In [ ]:
!pip -q install -U transformers accelerate datasets peft trl bitsandbytes sentencepiece scikit-learn pyyaml huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 164.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 140.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.6 MB/s eta 0:00:00


In [ ]:
# Reinstall datasets to fix potential pyarrow incompatibility
!pip uninstall -y datasets pyarrow
!pip install -q -U datasets pyarrow

Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1


In [ ]:
import os
import json
import math
import random
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter

import yaml
import torch



from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed,
)

from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model, PeftModel
from trl import SFTTrainer

ValueError: pyarrow.lib.IpcReadOptions size changed, may indicate binary incompatibility. Expected 112 from C header, got 104 from PyObject

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Hugging Face login

In [ ]:
from huggingface_hub import login

# You can paste a token interactively, or set an HF_TOKEN secret in Colab and use login(token=...)
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Config (DeepSeek run)


In [ ]:
@dataclass
class Config:
    DATA_PATH = "/content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl"
    OUTPUT_DIR = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b"
    LABEL_MAP_PATH = "/content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml"

    MODEL_ID: str = "deepseek-ai/deepseek-llm-7b-chat"
    FALLBACK_MODEL_ID: str = "deepseek-ai/deepseek-llm-7b-base"

    SEED: int = 42
    TEST_SIZE: float = 0.10
    VAL_SIZE: float = 0.10   # final split = train / val / test

    MAX_SEQ_LEN: int = 2048
    EPOCHS: int = 2
    LEARNING_RATE: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.03

    PER_DEVICE_TRAIN_BATCH_SIZE: int = 1
    PER_DEVICE_EVAL_BATCH_SIZE: int = 1
    GRADIENT_ACCUMULATION_STEPS: int = 8

    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05

    LOGGING_STEPS: int = 10
    SAVE_STEPS: int = 100
    EVAL_STEPS: int = 100

    USE_BF16: bool = True
    LOAD_IN_4BIT: bool = True
    RESUME_IF_CHECKPOINT_EXISTS: bool = True

cfg = Config()
set_seed(cfg.SEED)

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
os.makedirs(str(Path(cfg.LABEL_MAP_PATH).parent), exist_ok=True)

print(asdict(cfg))

{'MODEL_ID': 'deepseek-ai/deepseek-llm-7b-chat', 'FALLBACK_MODEL_ID': 'deepseek-ai/deepseek-llm-7b-base', 'SEED': 42, 'TEST_SIZE': 0.1, 'VAL_SIZE': 0.1, 'MAX_SEQ_LEN': 2048, 'EPOCHS': 2, 'LEARNING_RATE': 0.0002, 'WEIGHT_DECAY': 0.01, 'WARMUP_RATIO': 0.03, 'PER_DEVICE_TRAIN_BATCH_SIZE': 1, 'PER_DEVICE_EVAL_BATCH_SIZE': 1, 'GRADIENT_ACCUMULATION_STEPS': 8, 'LORA_R': 16, 'LORA_ALPHA': 32, 'LORA_DROPOUT': 0.05, 'LOGGING_STEPS': 10, 'SAVE_STEPS': 100, 'EVAL_STEPS': 100, 'USE_BF16': True, 'LOAD_IN_4BIT': True, 'RESUME_IF_CHECKPOINT_EXISTS': True}


verify dataset


In [ ]:
from pathlib import Path

path = Path(cfg.DATA_PATH)
print("Dataset path:", path)
print("Exists:", path.exists())

if not path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {cfg.DATA_PATH}. Upload the JSONL to Drive and update DATA_PATH if needed."
    )

with open(path, "r", encoding="utf-8") as f:
    line_count = sum(1 for _ in f)

print("Line count:", line_count)

Dataset path: /content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl
Exists: True
Line count: 7000


Load Jsonl

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {i}: {e}")
    return rows

rows = load_jsonl(cfg.DATA_PATH)
print(f"Loaded {len(rows):,} rows")
print("Sample keys:", sorted(rows[0].keys()))

Loaded 7,000 rows
Sample keys: ['claim_name', 'container_state', 'event_message', 'event_reason', 'evidence_text', 'image', 'last_state', 'namespace', 'node', 'node_selectors', 'pod_describe', 'pod_logs', 'pod_logs_previous', 'pod_name', 'pod_status', 'ready', 'restart_count', 'scenario_id', 'service_account_name']


Dataset inspection

In [ ]:
scenario_counts = Counter(r["scenario_id"] for r in rows)
evidence_lengths = [len((r.get("evidence_text") or "")) for r in rows]

print("Scenario frequencies:")
for k, v in sorted(scenario_counts.items()):
    print(f"  {k:45s} {v}")

print("\nEvidence length stats:")
print("  min:", min(evidence_lengths))
print("  max:", max(evidence_lengths))
print("  avg:", sum(evidence_lengths) / len(evidence_lengths))

required_scenarios = [
    "pvc_not_found_mountfail",
    "failedscheduling_taint",
    "dns_resolution_failure",
    "service_connection_refused",
    "createcontainerconfigerror_missing_secret",
    "createcontainerconfigerror_bad_configmap_key",
    "imagepull_bad_tag",
    "crashloop_bad_args",
    "oomkilled_limit_too_low",
    "imagepull_registry_auth",
    "failedscheduling_insufficient_memory",
    "failedscheduling_insufficient_cpu",
    "nodeselector_mismatch",
    "rbac_forbidden",
]

missing = sorted(set(required_scenarios) - set(scenario_counts.keys()))
extra = sorted(set(scenario_counts.keys()) - set(required_scenarios))

print("\nMissing expected scenarios:", missing)
print("Extra scenarios in file:", extra)

Scenario frequencies:
  crashloop_bad_args                            500
  createcontainerconfigerror_bad_configmap_key  500
  createcontainerconfigerror_missing_secret     500
  dns_resolution_failure                        500
  failedscheduling_insufficient_cpu             500
  failedscheduling_insufficient_memory          500
  failedscheduling_taint                        500
  imagepull_bad_tag                             500
  imagepull_registry_auth                       500
  nodeselector_mismatch                         500
  oomkilled_limit_too_low                       500
  pvc_not_found_mountfail                       512
  rbac_forbidden                                488
  service_connection_refused                    500

Evidence length stats:
  min: 1651
  max: 13602
  avg: 3764.0062857142857

Missing expected scenarios: []
Extra scenarios in file: []


Default RCA label map

In [ ]:
DEFAULT_SCENARIO_RCA = {
    "pvc_not_found_mountfail": (
        "The workload fails during volume setup because the referenced PersistentVolumeClaim does not exist or cannot be resolved in the namespace. "
        "Kubernetes cannot mount the requested storage, so the pod never starts normally.\n"
        "Remediation:\n"
        "- Verify the PVC name and namespace.\n"
        "- Create the missing PVC or fix the claim reference in the workload spec.\n"
        "- Confirm the bound volume is available before redeploying."
    ),
    "failedscheduling_taint": (
        "The pod is unschedulable because available nodes have taints that the pod does not tolerate. "
        "The scheduler rejects placement until matching tolerations are added or a compatible node is available.\n"
        "Remediation:\n"
        "- Inspect node taints with kubectl describe nodes.\n"
        "- Add the required tolerations to the pod spec if appropriate.\n"
        "- Alternatively schedule onto untainted nodes."
    ),
    "dns_resolution_failure": (
        "The application fails because in-cluster DNS resolution is not succeeding for the requested hostname or service name. "
        "As a result, the pod cannot resolve dependencies and the workload errors during startup or runtime.\n"
        "Remediation:\n"
        "- Check the target service name, namespace, and DNS suffix.\n"
        "- Verify CoreDNS and cluster DNS health.\n"
        "- Test name resolution from a debug pod."
    ),
    "service_connection_refused": (
        "The application reaches the target address but the connection is refused, which usually means the service endpoint is not accepting traffic on that port. "
        "This can happen when the backend is down, listening on a different port, or has no ready endpoints.\n"
        "Remediation:\n"
        "- Check whether the target service has ready endpoints.\n"
        "- Verify container port and service port mappings.\n"
        "- Confirm the backend process is actually listening."
    ),
    "createcontainerconfigerror_missing_secret": (
        "The pod enters CreateContainerConfigError because the container spec references a Secret that does not exist or is unavailable. "
        "Kubernetes cannot build the container configuration, so the container never starts.\n"
        "Remediation:\n"
        "- Verify the Secret name, key, and namespace.\n"
        "- Create the missing Secret or correct the reference.\n"
        "- Redeploy after confirming access."
    ),
    "createcontainerconfigerror_bad_configmap_key": (
        "The pod enters CreateContainerConfigError because it references a ConfigMap key that is missing or invalid. "
        "The container configuration cannot be completed with the requested configuration source.\n"
        "Remediation:\n"
        "- Check the ConfigMap name and referenced keys.\n"
        "- Add the missing key or correct the manifest reference.\n"
        "- Restart the workload after updating configuration."
    ),
    "imagepull_bad_tag": (
        "The container image cannot be pulled because the specified tag does not exist or is incorrect in the registry. "
        "The pod remains stuck waiting for a valid image to download.\n"
        "Remediation:\n"
        "- Verify the image repository and tag.\n"
        "- Update the deployment to a valid published tag.\n"
        "- Recreate the pod after fixing the image reference."
    ),
    "crashloop_bad_args": (
        "The container repeatedly crashes because the startup command or arguments are invalid for the image. "
        "The process exits quickly, causing Kubernetes to restart it and enter CrashLoopBackOff.\n"
        "Remediation:\n"
        "- Review the command and args configured in the pod spec.\n"
        "- Compare them with the image's expected entrypoint usage.\n"
        "- Fix the arguments and redeploy."
    ),
    "oomkilled_limit_too_low": (
        "The container is being OOMKilled because its memory limit is too low for the workload's actual usage. "
        "When the process exceeds the cgroup memory limit, the kernel terminates it and Kubernetes restarts the container.\n"
        "Remediation:\n"
        "- Increase the container memory limit and request.\n"
        "- Inspect memory usage patterns and leaks.\n"
        "- Re-run after adjusting resource settings."
    ),
    "imagepull_registry_auth": (
        "The image pull fails because the registry requires authentication and the pod lacks valid credentials. "
        "Without a working imagePullSecret or authorized service account, Kubernetes cannot download the image.\n"
        "Remediation:\n"
        "- Verify registry credentials and imagePullSecrets.\n"
        "- Ensure the secret exists in the correct namespace.\n"
        "- Attach the secret to the workload or service account."
    ),
    "failedscheduling_insufficient_memory": (
        "The pod cannot be scheduled because cluster nodes do not have enough allocatable memory to satisfy its request. "
        "The scheduler leaves the pod pending until capacity increases or requests are reduced.\n"
        "Remediation:\n"
        "- Reduce memory requests if they are oversized.\n"
        "- Free capacity or scale the node pool.\n"
        "- Check fragmentation and per-node allocatable memory."
    ),
    "failedscheduling_insufficient_cpu": (
        "The pod cannot be scheduled because cluster nodes do not have enough allocatable CPU to satisfy its request. "
        "The scheduler keeps the pod pending until sufficient CPU is available.\n"
        "Remediation:\n"
        "- Reduce CPU requests if possible.\n"
        "- Scale the cluster or free existing CPU reservations.\n"
        "- Review bin-packing and node capacity."
    ),
    "nodeselector_mismatch": (
        "The pod is unschedulable because its nodeSelector does not match labels on any available node. "
        "The scheduler cannot place the workload until the selector or node labels are corrected.\n"
        "Remediation:\n"
        "- Inspect the nodeSelector in the workload spec.\n"
        "- Compare it against actual node labels.\n"
        "- Fix the selector or label the intended nodes."
    ),
    "rbac_forbidden": (
        "The workload fails because the service account does not have RBAC permission to perform the requested Kubernetes API action. "
        "The API server returns Forbidden, blocking the application from accessing cluster resources it expects.\n"
        "Remediation:\n"
        "- Identify the denied verb, resource, and API group.\n"
        "- Update the Role or ClusterRole and corresponding binding.\n"
        "- Verify the pod is using the intended service account."
    ),
}

print(f"Default RCA labels: {len(DEFAULT_SCENARIO_RCA)} scenarios")
print(sorted(DEFAULT_SCENARIO_RCA.keys()))
print()
print(DEFAULT_SCENARIO_RCA["crashloop_bad_args"])

Default RCA labels: 14 scenarios
['crashloop_bad_args', 'createcontainerconfigerror_bad_configmap_key', 'createcontainerconfigerror_missing_secret', 'dns_resolution_failure', 'failedscheduling_insufficient_cpu', 'failedscheduling_insufficient_memory', 'failedscheduling_taint', 'imagepull_bad_tag', 'imagepull_registry_auth', 'nodeselector_mismatch', 'oomkilled_limit_too_low', 'pvc_not_found_mountfail', 'rbac_forbidden', 'service_connection_refused']

The container repeatedly crashes because the startup command or arguments are invalid for the image. The process exits quickly, causing Kubernetes to restart it and enter CrashLoopBackOff.
Remediation:
- Review the command and args configured in the pod spec.
- Compare them with the image's expected entrypoint usage.
- Fix the arguments and redeploy.


In [ ]:
from pathlib import Path
import json
import yaml

def load_scenario_rca(label_map_path, default_map):
    label_path = Path(label_map_path)

    # Start from defaults no matter what
    merged = dict(default_map)

    if label_path.exists():
        print(f"Loading scenario RCA map from {label_path}")
        with open(label_path, "r", encoding="utf-8") as f:
            if label_path.suffix.lower() in {".yaml", ".yml"}:
                loaded = yaml.safe_load(f)
            else:
                loaded = json.load(f)

        if loaded is None:
            loaded = {}

        if not isinstance(loaded, dict):
            raise ValueError("scenario_rca file must contain a dictionary mapping scenario_id -> RCA text")

        # Override defaults with anything present in file
        merged.update(loaded)
    else:
        print("No external label file found. Writing default YAML template to Drive.")
        with open(label_path, "w", encoding="utf-8") as f:
            yaml.safe_dump(default_map, f, sort_keys=True, allow_unicode=True)

    # Backfill any missing keys into the YAML on disk too
    missing_in_file = [k for k in default_map if k not in merged]
    if missing_in_file:
        print("Backfilling missing keys:", missing_in_file)
        for k in missing_in_file:
            merged[k] = default_map[k]

    with open(label_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(merged, f, sort_keys=True, allow_unicode=True)

    return merged

SCENARIO_RCA = load_scenario_rca(cfg.LABEL_MAP_PATH, DEFAULT_SCENARIO_RCA)

missing_keys = sorted(set(required_scenarios) - set(SCENARIO_RCA.keys()))
if missing_keys:
    raise ValueError(f"SCENARIO_RCA missing keys even after backfill: {missing_keys}")

print("Loaded RCA label map with", len(SCENARIO_RCA), "entries")
print("Keys:")
for k in sorted(SCENARIO_RCA.keys()):
    print("-", k)

Loading scenario RCA map from /content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml
Loaded RCA label map with 14 entries
Keys:
- crashloop_bad_args
- createcontainerconfigerror_bad_configmap_key
- createcontainerconfigerror_missing_secret
- dns_resolution_failure
- failedscheduling_insufficient_cpu
- failedscheduling_insufficient_memory
- failedscheduling_taint
- imagepull_bad_tag
- imagepull_registry_auth
- nodeselector_mismatch
- oomkilled_limit_too_low
- pvc_not_found_mountfail
- rbac_forbidden
- service_connection_refused


Build prompt formatting

In [ ]:
SYSTEM_PROMPT = (
    "You are a Kubernetes SRE performing root cause analysis. "
    "Given pod and event evidence, produce a concise RCA with: "
    "(1) what failed, (2) the likely root cause in Kubernetes terms, "
    "and (3) 2 to 4 short remediation bullets."
)


def compact_evidence(row):
    evidence = (row.get("evidence_text") or "").strip()
    namespace = row.get("namespace", "")
    pod_name = row.get("pod_name", "")
    pod_status = row.get("pod_status", "")
    event_reason = row.get("event_reason", "")
    event_message = row.get("event_message", "")

    header = (
        f"Namespace: {namespace}\n"
        f"Pod: {pod_name}\n"
        f"Pod status: {pod_status}\n"
        f"Event reason: {event_reason}\n"
        f"Event message: {event_message}\n"
    )

    return header + "\nEvidence:\n" + evidence


def build_messages(row):
    user_text = (
        "Analyze the following Kubernetes incident evidence and provide the RCA.\n\n"
        + compact_evidence(row)
    )
    assistant_text = SCENARIO_RCA[row["scenario_id"]]
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": assistant_text},
    ]


def build_inference_messages(row):
    user_text = (
        "Analyze the following Kubernetes incident evidence and provide the RCA.\n\n"
        + compact_evidence(row)
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]

sample_messages = build_messages(rows[0])
print(sample_messages[0]["content"])
print()
print(sample_messages[1]["content"][:1000])
print()
print(sample_messages[2]["content"])

You are a Kubernetes SRE performing root cause analysis. Given pod and event evidence, produce a concise RCA with: (1) what failed, (2) the likely root cause in Kubernetes terms, and (3) 2 to 4 short remediation bullets.

Analyze the following Kubernetes incident evidence and provide the RCA.

Namespace: data-pipeline
Pod: busybox-pn3zirfh-86bbb7957c-7754l
Pod status: Pending
Event reason: FailedScheduling
Event message: 0/1 nodes are available: persistentvolumeclaim "missing-pvc-pn3zirfh" not found. preemption: 0/1 nodes are available: 1 Preemption is not helpful for scheduling.

Evidence:
POD DESCRIBE:
Name:             busybox-pn3zirfh-86bbb7957c-7754l
Namespace:        data-pipeline
Priority:         0
Service Account:  default
Node:             <none>
Labels:           app=busybox-pn3zirfh
                  pod-template-hash=86bbb7957c
                  run_id=pn3zirfh
                  scenario_id=pvc_not_found_mountfail
                  synthetic=true
Annotations:      <none>
S

Stratified train/val/test split

In [ ]:
indices = list(range(len(rows)))
labels = [r["scenario_id"] for r in rows]

train_idx, test_idx = train_test_split(
    indices,
    test_size=cfg.TEST_SIZE,
    random_state=cfg.SEED,
    stratify=labels,
)

train_labels = [labels[i] for i in train_idx]
val_relative = cfg.VAL_SIZE / (1.0 - cfg.TEST_SIZE)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=val_relative,
    random_state=cfg.SEED,
    stratify=train_labels,
)

train_rows = [rows[i] for i in train_idx]
val_rows = [rows[i] for i in val_idx]
test_rows = [rows[i] for i in test_idx]

print(f"Train: {len(train_rows)}")
print(f"Val:   {len(val_rows)}")
print(f"Test:  {len(test_rows)}")

for name, split_rows in [("train", train_rows), ("val", val_rows), ("test", test_rows)]:
    c = Counter(r["scenario_id"] for r in split_rows)
    print(f"\n{name} scenario distribution:")
    for k, v in sorted(c.items()):
        print(f"  {k:45s} {v}")

Train: 5600
Val:   700
Test:  700

train scenario distribution:
  crashloop_bad_args                            400
  createcontainerconfigerror_bad_configmap_key  400
  createcontainerconfigerror_missing_secret     400
  dns_resolution_failure                        400
  failedscheduling_insufficient_cpu             400
  failedscheduling_insufficient_memory          400
  failedscheduling_taint                        400
  imagepull_bad_tag                             400
  imagepull_registry_auth                       400
  nodeselector_mismatch                         400
  oomkilled_limit_too_low                       400
  pvc_not_found_mountfail                       410
  rbac_forbidden                                390
  service_connection_refused                    400

val scenario distribution:
  crashloop_bad_args                            50
  createcontainerconfigerror_bad_configmap_key  50
  createcontainerconfigerror_missing_secret     50
  dns_resolution_failure   

Convert to Hugging Face Datasets

In [ ]:
def to_chat_record(row):
    return {
        "scenario_id": row["scenario_id"],
        "messages": build_messages(row),
        "inference_messages": build_inference_messages(row),
        "evidence_text": row.get("evidence_text", ""),
    }

train_ds = Dataset.from_list([to_chat_record(r) for r in train_rows])
val_ds = Dataset.from_list([to_chat_record(r) for r in val_rows])
test_ds = Dataset.from_list([to_chat_record(r) for r in test_rows])

print(train_ds)
print(val_ds)
print(test_ds)

Dataset({
    features: ['scenario_id', 'messages', 'inference_messages', 'evidence_text'],
    num_rows: 5600
})
Dataset({
    features: ['scenario_id', 'messages', 'inference_messages', 'evidence_text'],
    num_rows: 700
})
Dataset({
    features: ['scenario_id', 'messages', 'inference_messages', 'evidence_text'],
    num_rows: 700
})


Load DeepSeek tokenizer

In [ ]:
model_id_to_use = cfg.MODEL_ID

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id_to_use, trust_remote_code=True)
except Exception as e:
    print(f"Failed to load {cfg.MODEL_ID}: {e}")
    print(f"Falling back to {cfg.FALLBACK_MODEL_ID}")
    model_id_to_use = cfg.FALLBACK_MODEL_ID
    tokenizer = AutoTokenizer.from_pretrained(model_id_to_use, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"
print("Using model:", model_id_to_use)
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Using model: deepseek-ai/deepseek-llm-7b-chat
Pad token: <｜end▁of▁sentence｜>


Format chat examples into plain text for SFTTrainer

In [ ]:
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds_text = train_ds.map(apply_chat_template)
val_ds_text = val_ds.map(apply_chat_template)

print(train_ds_text[0]["text"][:2000])

Map:   0%|          | 0/5600 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

<｜begin▁of▁sentence｜>You are a Kubernetes SRE performing root cause analysis. Given pod and event evidence, produce a concise RCA with: (1) what failed, (2) the likely root cause in Kubernetes terms, and (3) 2 to 4 short remediation bullets.

User: Analyze the following Kubernetes incident evidence and provide the RCA.

Namespace: team-a-dev-v8rq8p
Pod: web-w-9cj8-0
Pod status: Pending
Event reason: Scheduled
Event message: Successfully assigned team-a-dev-v8rq8p/web-w-9cj8-0 to k8s-incidents-control-plane

Evidence:
namespace: team-a-dev-v8rq8p
workload: web-w-9cj8
container: sidecar
image: docker.io/library/nginx:2.0.1
=== kubectl get pods ===
NAME           READY   STATUS         RESTARTS   AGE
web-w-9cj8-0   0/1     ErrImagePull   0          5s
=== kubectl describe pod ===
Name:             web-w-9cj8-0
Namespace:        team-a-dev-v8rq8p
Priority:         0
Service Account:  default
Node:             k8s-incidents-control-plane/172.22.0.2
Start Time:       Mon, 30 Mar 2026 20:17:4

Token length sanity check

In [ ]:
def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])

sample_lengths = [count_tokens(train_ds_text[i]["text"]) for i in range(min(200, len(train_ds_text)))]
print("Sample token lengths over first 200 train rows:")
print("  min:", min(sample_lengths))
print("  max:", max(sample_lengths))
print("  avg:", sum(sample_lengths) / len(sample_lengths))
print("Configured MAX_SEQ_LEN:", cfg.MAX_SEQ_LEN)

Token indices sequence length is longer than the specified maximum sequence length for this model (4977 > 4096). Running this sequence through the model will result in indexing errors


Sample token lengths over first 200 train rows:
  min: 602
  max: 5274
  avg: 1314.02
Configured MAX_SEQ_LEN: 2048


4-bit quantization + LoRA config

In [ ]:
compute_dtype = torch.bfloat16 if cfg.USE_BF16 and torch.cuda.is_available() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=cfg.LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

peft_config = LoraConfig(
    r=cfg.LORA_R,
    lora_alpha=cfg.LORA_ALPHA,
    lora_dropout=cfg.LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
)

print(bnb_config)
print(peft_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "bfloat16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=16, target_modules={'k_proj', 'q_proj', 'gate_proj', 'down_proj', 'up_proj', 'o_proj', 'v_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='m

Load base model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id_to_use,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

trainable params: 37,478,400 || all params: 6,947,844,096 || trainable%: 0.5394


Detect latest checkpoint for resume

In [ ]:
def get_latest_checkpoint(output_dir):
    output_path = Path(output_dir)
    if not output_path.exists():
        return None

    checkpoints = []
    for p in output_path.iterdir():
        if p.is_dir() and p.name.startswith("checkpoint-"):
            try:
                step = int(p.name.split("-")[-1])
                checkpoints.append((step, str(p)))
            except ValueError:
                pass

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]

latest_checkpoint = get_latest_checkpoint(cfg.OUTPUT_DIR)
print("Latest checkpoint:", latest_checkpoint)

Latest checkpoint: None


Training arguments

In [ ]:
from trl import SFTConfig

sft_args = SFTConfig(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
    warmup_ratio=cfg.WARMUP_RATIO,
    logging_steps=cfg.LOGGING_STEPS,
    save_steps=cfg.SAVE_STEPS,
    eval_steps=cfg.EVAL_STEPS,
    save_strategy="steps",
    eval_strategy="steps",
    save_total_limit=3,
    bf16=cfg.USE_BF16,
    fp16=not cfg.USE_BF16,
    report_to="none",
    lr_scheduler_type="cosine",
    seed=cfg.SEED,
    dataloader_pin_memory=False,
    max_length=cfg.MAX_SEQ_LEN,
    packing=False,
)
print(sft_args)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=False,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=False,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eos_tok

Trainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds_text,
    eval_dataset=val_ds_text,
    args=sft_args,
    processing_class=tokenizer,
)

Tokenizing train dataset:   0%|          | 0/5600 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/700 [00:00<?, ? examples/s]

Train with automatic resume

In [ ]:
resume_path = latest_checkpoint if (cfg.RESUME_IF_CHECKPOINT_EXISTS and latest_checkpoint is not None) else None
print("Resuming from:", resume_path)

train_result = trainer.train(resume_from_checkpoint=resume_path)
trainer.save_model(cfg.OUTPUT_DIR)
tokenizer.save_pretrained(cfg.OUTPUT_DIR)

print(train_result)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 100001}.


Resuming from: None


Step,Training Loss,Validation Loss
100,0.200987,0.209498
200,0.196323,0.180797
300,0.175325,0.178267
400,0.172302,0.177114
500,0.175911,0.175662
600,0.192562,0.175084
700,0.176663,0.174543
800,0.177702,0.174208
900,0.190301,0.173724
1000,0.192770,0.173504


TrainOutput(global_step=1400, training_loss=0.21989068167550224, metrics={'train_runtime': 32040.2156, 'train_samples_per_second': 0.35, 'train_steps_per_second': 0.044, 'total_flos': 5.371875962460242e+17, 'train_loss': 0.21989068167550224})


Save run metadata

In [ ]:
run_info = {
    "config": asdict(cfg),
    "model_id_used": model_id_to_use,
    "latest_checkpoint_before_train": latest_checkpoint,
    "train_size": len(train_rows),
    "val_size": len(val_rows),
    "test_size": len(test_rows),
}

with open(os.path.join(cfg.OUTPUT_DIR, "run_info.json"), "w", encoding="utf-8") as f:
    json.dump(run_info, f, indent=2)

print(json.dumps(run_info, indent=2))

{
  "config": {
    "MODEL_ID": "deepseek-ai/deepseek-llm-7b-chat",
    "FALLBACK_MODEL_ID": "deepseek-ai/deepseek-llm-7b-base",
    "SEED": 42,
    "TEST_SIZE": 0.1,
    "VAL_SIZE": 0.1,
    "MAX_SEQ_LEN": 2048,
    "EPOCHS": 2,
    "LEARNING_RATE": 0.0002,
    "WEIGHT_DECAY": 0.01,
    "WARMUP_RATIO": 0.03,
    "PER_DEVICE_TRAIN_BATCH_SIZE": 1,
    "PER_DEVICE_EVAL_BATCH_SIZE": 1,
    "GRADIENT_ACCUMULATION_STEPS": 8,
    "LORA_R": 16,
    "LORA_ALPHA": 32,
    "LORA_DROPOUT": 0.05,
    "LOGGING_STEPS": 10,
    "SAVE_STEPS": 100,
    "EVAL_STEPS": 100,
    "USE_BF16": true,
    "LOAD_IN_4BIT": true,
    "RESUME_IF_CHECKPOINT_EXISTS": true
  },
  "model_id_used": "deepseek-ai/deepseek-llm-7b-chat",
  "latest_checkpoint_before_train": null,
  "train_size": 5600,
  "val_size": 700,
  "test_size": 700
}


Validation loss / safe post-train evaluation

In [ ]:
from transformers.utils.notebook import NotebookProgressCallback

# This avoids the notebook callback issue some Colab/Trainer versions hit after train()
try:
    trainer.remove_callback(NotebookProgressCallback)
except Exception as e:
    print("Callback removal warning:", e)

try:
    eval_metrics = trainer.evaluate()
    print(eval_metrics)
except Exception as e:
    print("Standalone trainer.evaluate() failed:", e)
    print("Use the validation loss logged during training as the primary eval signal for this run.")

{'eval_loss': 0.17319004237651825, 'eval_runtime': 489.4205, 'eval_samples_per_second': 1.43, 'eval_steps_per_second': 1.43}


Reload adapter for inference

In [ ]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Free training objects before loading inference model
try:
    del trainer
except NameError:
    pass

try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

inference_base = AutoModelForCausalLM.from_pretrained(
    model_id_to_use,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

inference_model = PeftModel.from_pretrained(inference_base, cfg.OUTPUT_DIR)
inference_model.eval()

print("Inference model loaded successfully.")

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

Inference model loaded successfully.


Run one validation example

In [ ]:
sample_idx = 0
sample = val_rows[sample_idx]
messages = build_inference_messages(sample)

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

device = next(inference_model.parameters()).device

inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    truncation=True,
    max_length=cfg.MAX_SEQ_LEN,
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=220,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

completion = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=== TRUE SCENARIO ===")
print(sample["scenario_id"])
print("\n=== PROMPT ===")
print(prompt_text[:4000])
print("\n=== MODEL OUTPUT ===")
print(completion)
print("\n=== TARGET RCA ===")
print(SCENARIO_RCA[sample["scenario_id"]])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== TRUE SCENARIO ===
dns_resolution_failure

=== PROMPT ===
<｜begin▁of▁sentence｜>You are a Kubernetes SRE performing root cause analysis. Given pod and event evidence, produce a concise RCA with: (1) what failed, (2) the likely root cause in Kubernetes terms, and (3) 2 to 4 short remediation bullets.

User: Analyze the following Kubernetes incident evidence and provide the RCA.

Namespace: mk-dc327p27
Pod: dns-fail-demo-l6bcg
Pod status: Failed
Event reason: Scheduled
Event message: Successfully assigned mk-dc327p27/dns-fail-demo-l6bcg to docker-desktop

Evidence:
POD DESCRIBE:
Name:             dns-fail-demo-l6bcg
Namespace:        mk-dc327p27
Priority:         0
Service Account:  default
Node:             docker-desktop/192.168.65.3
Start Time:       Sun, 29 Mar 2026 00:30:54 -0700
Labels:           batch.kubernetes.io/controller-uid=38314785-4373-4a35-acf5-6a145f551d92
                  batch.kubernetes.io/job-name=dns-fail-demo
                  controller-uid=38314785-4373-4a35-a

Print a few fixed validation generations

In [ ]:
fixed_indices = [0, 1, 2, 3, 4]
device = next(inference_model.parameters()).device

for idx in fixed_indices:
    sample = val_rows[idx]
    messages = build_inference_messages(sample)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            temperature=0.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("=" * 100)
    print("Scenario:", sample["scenario_id"])
    print("Evidence preview:")
    evidence = sample.get("evidence_text", "")
    print((evidence[:700] + "...") if len(evidence) > 700 else evidence)
    print("\nModel output:")
    print(completion)
    print("\nTarget:")
    print(SCENARIO_RCA[sample["scenario_id"]])
    print()

Scenario: dns_resolution_failure
Evidence preview:
POD DESCRIBE:
Name:             dns-fail-demo-l6bcg
Namespace:        mk-dc327p27
Priority:         0
Service Account:  default
Node:             docker-desktop/192.168.65.3
Start Time:       Sun, 29 Mar 2026 00:30:54 -0700
Labels:           batch.kubernetes.io/controller-uid=38314785-4373-4a35-acf5-6a145f551d92
                  batch.kubernetes.io/job-name=dns-fail-demo
                  controller-uid=38314785-4373-4a35-acf5-6a145f551d92
                  job-name=dns-fail-demo
Annotations:      <none>
Status:           Failed
IP:               10.1.2.64
IPs:
  IP:           10.1.2.64
Controlled By:  Job/dns-fail-demo
Containers:
  dns-test:
    Container ID:  docker://d22ac266a7845776f06...

Model output:
Theapplicationfailsbecausein-clusterDNSresolutionisnotsucceedingfortherequestedhostnameorservicename.Asaresult,thepodcannotresolvedependenciesandtheworkloaderrorsduringstartuporruntime.Remediation:-Checkthetargetservicename,namesp

evaluation


In [ ]:
def predict_scenario_from_output(text):
    text = text.lower()

    rules = {
        "oomkilled_limit_too_low": ["oom", "memory limit"],
        "crashloop_bad_args": ["crashloop", "arguments", "restart"],
        "imagepull_bad_tag": ["image", "tag", "pull"],
        "imagepull_registry_auth": ["authentication", "pull secret", "credentials"],
        "pvc_not_found_mountfail": ["pvc", "volume", "mount"],
        "failedscheduling_taint": ["taint", "toleration"],
        "failedscheduling_insufficient_memory": ["insufficient memory"],
        "failedscheduling_insufficient_cpu": ["insufficient cpu"],
        "nodeselector_mismatch": ["nodeselector", "label"],
        "rbac_forbidden": ["rbac", "forbidden", "permission"],
        "dns_resolution_failure": ["dns", "resolve"],
        "service_connection_refused": ["connection refused"],
        "createcontainerconfigerror_missing_secret": ["secret"],
        "createcontainerconfigerror_bad_configmap_key": ["configmap"],
    }

    for scenario, keywords in rules.items():
        for kw in keywords:
            if kw in text:
                return scenario

    return "unknown"

In [ ]:
correct = 0
total = 100  # evaluate on first 100 examples

device = next(inference_model.parameters()).device

for i in range(total):
    sample = val_rows[i]
    messages = build_inference_messages(sample)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=0.0,
        )

    pred_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    pred_scenario = predict_scenario_from_output(pred_text)
    true_scenario = sample["scenario_id"]

    if pred_scenario == true_scenario:
        correct += 1

accuracy = correct / total
print(f"Scenario Accuracy (proxy): {accuracy:.3f}")

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for

Scenario Accuracy (proxy): 0.590


In [ ]:
from collections import defaultdict

scenario_correct = defaultdict(int)
scenario_total = defaultdict(int)

total = min(100, len(val_rows))
device = next(inference_model.parameters()).device

for i in range(total):
    sample = val_rows[i]
    messages = build_inference_messages(sample)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=0.0,
        )

    pred_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    pred_scenario = predict_scenario_from_output(pred_text)
    true_scenario = sample["scenario_id"]

    scenario_total[true_scenario] += 1
    if pred_scenario == true_scenario:
        scenario_correct[true_scenario] += 1

print("Per-scenario accuracy:")
for scenario in sorted(scenario_total):
    acc = scenario_correct[scenario] / scenario_total[scenario]
    print(f"{scenario:45s} {acc:.3f}  ({scenario_correct[scenario]}/{scenario_total[scenario]})")

Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:100001 for

Per-scenario accuracy:
crashloop_bad_args                            1.000  (6/6)
createcontainerconfigerror_bad_configmap_key  0.000  (0/5)
createcontainerconfigerror_missing_secret     1.000  (8/8)
dns_resolution_failure                        1.000  (9/9)
failedscheduling_insufficient_cpu             0.000  (0/9)
failedscheduling_insufficient_memory          0.000  (0/3)
failedscheduling_taint                        1.000  (5/5)
imagepull_bad_tag                             1.000  (11/11)
imagepull_registry_auth                       0.000  (0/8)
nodeselector_mismatch                         0.000  (0/4)
oomkilled_limit_too_low                       1.000  (6/6)
pvc_not_found_mountfail                       1.000  (9/9)
rbac_forbidden                                1.000  (5/5)
service_connection_refused                    0.000  (0/12)


In [ ]:
print("Per-scenario accuracy:")
for scenario in sorted(scenario_total):
    acc = scenario_correct[scenario] / scenario_total[scenario]
    print(f"{scenario:45s} {acc*100:.2f}%  ({scenario_correct[scenario]}/{scenario_total[scenario]})")

Per-scenario accuracy:
crashloop_bad_args                            100.00%  (6/6)
createcontainerconfigerror_bad_configmap_key  0.00%  (0/5)
createcontainerconfigerror_missing_secret     100.00%  (8/8)
dns_resolution_failure                        100.00%  (9/9)
failedscheduling_insufficient_cpu             0.00%  (0/9)
failedscheduling_insufficient_memory          0.00%  (0/3)
failedscheduling_taint                        100.00%  (5/5)
imagepull_bad_tag                             100.00%  (11/11)
imagepull_registry_auth                       0.00%  (0/8)
nodeselector_mismatch                         0.00%  (0/4)
oomkilled_limit_too_low                       100.00%  (6/6)
pvc_not_found_mountfail                       100.00%  (9/9)
rbac_forbidden                                100.00%  (5/5)
service_connection_refused                    0.00%  (0/12)


In [ ]:
accuracy = correct / total
print(f"Scenario Accuracy: {accuracy*100:.2f}%")

Scenario Accuracy: 59.00%


In [ ]:
QWEN_PATH = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/qwen_rca_8b"
DEEPSEEK_PATH = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b"

In [ ]:
!pip install -q transformers peft accelerate datasets

In [ ]:
!pip uninstall -y scikit-learn
!pip install -q --no-cache-dir scikit-learn

Found existing installation: scikit-learn 1.8.0
Uninstalling scikit-learn-1.8.0:
  Successfully uninstalled scikit-learn-1.8.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 150.4 MB/s eta 0:00:00


In [ ]:
import os
import gc
import json
import yaml
import random
import torch

from dataclasses import dataclass
from collections import defaultdict

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

In [ ]:
@dataclass
class EvalConfig:
    DATA_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl"
    LABEL_MAP_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml"

    QWEN_MODEL_ID: str = "Qwen/Qwen2.5-7B-Instruct"
    DEEPSEEK_MODEL_ID: str = "deepseek-ai/deepseek-llm-7b-chat"

    QWEN_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/qwen_rca_8b"
    DEEPSEEK_PATH: str = "/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b"

    SEED: int = 42
    TEST_SIZE: float = 0.10
    VAL_SIZE: float = 0.10
    MAX_SEQ_LEN: int = 2048
    EVAL_SAMPLES: int = 700

cfg = EvalConfig()
set_seed(cfg.SEED)

print(cfg)

EvalConfig(DATA_PATH='/content/drive/MyDrive/Data298ModelTraining/k8s_combined_incidents.jsonl', LABEL_MAP_PATH='/content/drive/MyDrive/Data298ModelTraining/scenario_rca.yaml', QWEN_MODEL_ID='Qwen/Qwen2.5-7B-Instruct', DEEPSEEK_MODEL_ID='deepseek-ai/deepseek-llm-7b-chat', QWEN_PATH='/content/drive/MyDrive/Data298ModelTraining/checkpoints/qwen_rca_8b', DEEPSEEK_PATH='/content/drive/MyDrive/Data298ModelTraining/checkpoints/deepseek_rca_7b', SEED=42, TEST_SIZE=0.1, VAL_SIZE=0.1, MAX_SEQ_LEN=2048, EVAL_SAMPLES=700)


In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

rows = load_jsonl(cfg.DATA_PATH)

print("Loaded rows:", len(rows))
print("First row keys:", rows[0].keys())
print("First scenario:", rows[0]["scenario_id"])

Loaded rows: 7000
First row keys: dict_keys(['scenario_id', 'namespace', 'pod_name', 'service_account_name', 'node', 'pod_status', 'image', 'container_state', 'last_state', 'ready', 'restart_count', 'node_selectors', 'claim_name', 'event_reason', 'event_message', 'pod_describe', 'pod_logs', 'pod_logs_previous', 'evidence_text'])
First scenario: pvc_not_found_mountfail


In [ ]:
with open(cfg.LABEL_MAP_PATH, "r", encoding="utf-8") as f:
    SCENARIO_RCA = yaml.safe_load(f)

print("Loaded RCA labels:", len(SCENARIO_RCA))

Loaded RCA labels: 14


In [ ]:
def stratified_split(rows, test_size=0.10, val_size=0.10, seed=42):
    rng = random.Random(seed)

    by_label = defaultdict(list)
    for row in rows:
        by_label[row["scenario_id"]].append(row)

    train_rows, val_rows, test_rows = [], [], []

    for label, items in by_label.items():
        items = items[:]
        rng.shuffle(items)

        n = len(items)
        n_test = max(1, round(n * test_size))
        n_val = max(1, round(n * val_size))

        # avoid overflow for small classes
        if n_test + n_val >= n:
            if n >= 3:
                n_test = 1
                n_val = 1
            elif n == 2:
                n_test = 1
                n_val = 0
            else:
                n_test = 0
                n_val = 0

        test_part = items[:n_test]
        val_part = items[n_test:n_test + n_val]
        train_part = items[n_test + n_val:]

        test_rows.extend(test_part)
        val_rows.extend(val_part)
        train_rows.extend(train_part)

    rng.shuffle(train_rows)
    rng.shuffle(val_rows)
    rng.shuffle(test_rows)

    return train_rows, val_rows, test_rows

train_rows, val_rows, test_rows = stratified_split(
    rows,
    test_size=cfg.TEST_SIZE,
    val_size=cfg.VAL_SIZE,
    seed=cfg.SEED,
)

print(f"Train: {len(train_rows)}")
print(f"Val:   {len(val_rows)}")
print(f"Test:  {len(test_rows)}")
print("First val scenario:", val_rows[0]["scenario_id"])

Train: 5600
Val:   700
Test:  700
First val scenario: dns_resolution_failure


In [ ]:
SYSTEM_PROMPT = (
    "You are a Kubernetes SRE performing root cause analysis. "
    "Given pod and event evidence, produce a concise RCA with "
    "(1) what failed, (2) the likely root cause in Kubernetes terms, "
    "and (3) 2 to 4 short remediation bullets."
)

def compact_evidence(row):
    evidence = (row.get("evidence_text") or "").strip()
    namespace = row.get("namespace", "")
    pod_name = row.get("pod_name", "")
    pod_status = row.get("pod_status", "")
    event_reason = row.get("event_reason", "")
    event_message = row.get("event_message", "")

    return (
        f"Namespace: {namespace}\n"
        f"Pod: {pod_name}\n"
        f"Pod status: {pod_status}\n"
        f"Event reason: {event_reason}\n"
        f"Event message: {event_message}\n\n"
        f"Evidence:\n{evidence}"
    )

def build_inference_messages(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": "Analyze the following Kubernetes incident and provide the RCA.\n\n"
            + compact_evidence(row),
        },
    ]

In [ ]:
def predict_scenario_from_output(text):
    text = text.lower()

    rules = {
        "oomkilled_limit_too_low": ["oom", "memory limit"],
        "crashloop_bad_args": ["crashloop", "argument", "restart"],
        "imagepull_bad_tag": ["image", "tag"],
        "imagepull_registry_auth": ["auth", "credential", "unauthorized"],
        "pvc_not_found_mountfail": ["pvc", "volume", "mount"],
        "failedscheduling_taint": ["taint", "toleration"],
        "failedscheduling_insufficient_memory": ["insufficient memory"],
        "failedscheduling_insufficient_cpu": ["insufficient cpu"],
        "nodeselector_mismatch": ["nodeselector", "label"],
        "rbac_forbidden": ["rbac", "forbidden", "permission"],
        "dns_resolution_failure": ["dns", "resolve"],
        "service_connection_refused": ["connection refused", "refused"],
        "createcontainerconfigerror_missing_secret": ["secret"],
        "createcontainerconfigerror_bad_configmap_key": ["configmap"],
    }

    for scenario, keywords in rules.items():
        for kw in keywords:
            if kw in text:
                return scenario

    return "unknown"

In [ ]:
def resolve_adapter_path(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Path does not exist: {path}")

    if os.path.exists(os.path.join(path, "adapter_config.json")):
        return path

    candidates = []
    for name in os.listdir(path):
        full = os.path.join(path, name)
        if os.path.isdir(full) and name.startswith("checkpoint-"):
            if os.path.exists(os.path.join(full, "adapter_config.json")):
                step = int(name.split("-")[-1])
                candidates.append((step, full))

    if candidates:
        candidates.sort(key=lambda x: x[0])
        best = candidates[-1][1]
        print("Resolved adapter path:", best)
        return best

    raise ValueError(f"No adapter_config.json found in {path} or checkpoint-* subfolders")

In [ ]:
def load_model(model_id, adapter_root_path):
    adapter_path = resolve_adapter_path(adapter_root_path)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    model.config.pad_token_id = tokenizer.pad_token_id
    if hasattr(model, "generation_config") and model.generation_config is not None:
        model.generation_config.pad_token_id = tokenizer.pad_token_id
        model.generation_config.eos_token_id = tokenizer.eos_token_id

    return model, tokenizer

In [ ]:
def evaluate_model(model, tokenizer, name, val_rows, eval_samples=700, max_seq_len=2048):
    device = next(model.parameters()).device

    total = min(eval_samples, len(val_rows))
    correct = 0
    unknown = 0

    print(f"\nEvaluating {name} on {total} validation samples...")

    for i in range(total):
        sample = val_rows[i]
        messages = build_inference_messages(sample)

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_seq_len,
            padding=False,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        pred_text = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        pred = predict_scenario_from_output(pred_text)
        true = sample["scenario_id"]

        if pred == "unknown":
            unknown += 1
        if pred == true:
            correct += 1

        if (i + 1) % 50 == 0:
            print(f"{name}: processed {i + 1}/{total}")

    acc = correct / total

    print(f"\n{name} Results")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(f"Correct:  {correct}/{total}")
    print(f"Unknown:  {unknown}")

    return {
        "name": name,
        "accuracy": acc,
        "correct": correct,
        "total": total,
        "unknown": unknown,
    }

In [ ]:
def cleanup_model(model, tokenizer):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
print("Rows:", len(rows))
print("Val rows:", len(val_rows))
print("Qwen path exists:", os.path.exists(cfg.QWEN_PATH))
print("DeepSeek path exists:", os.path.exists(cfg.DEEPSEEK_PATH))

Rows: 7000
Val rows: 700
Qwen path exists: True
DeepSeek path exists: True


In [ ]:
print("Loading Qwen...")
qwen_model, qwen_tokenizer = load_model(cfg.QWEN_MODEL_ID, cfg.QWEN_PATH)

qwen_metrics = evaluate_model(
    qwen_model,
    qwen_tokenizer,
    "Qwen",
    val_rows,
    eval_samples=cfg.EVAL_SAMPLES,
    max_seq_len=cfg.MAX_SEQ_LEN,
)

cleanup_model(qwen_model, qwen_tokenizer)

Loading Qwen...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Evaluating Qwen on 700 validation samples...
Qwen: processed 50/700
Qwen: processed 100/700
Qwen: processed 150/700
Qwen: processed 200/700
Qwen: processed 250/700
Qwen: processed 300/700
Qwen: processed 350/700
Qwen: processed 400/700
Qwen: processed 450/700
Qwen: processed 500/700
Qwen: processed 550/700
Qwen: processed 600/700
Qwen: processed 650/700
Qwen: processed 700/700

Qwen Results
Accuracy: 70.86%
Correct:  496/700
Unknown:  101


In [ ]:
# 1) Clear models from memory
import gc
import torch

for name in ["qwen_model", "qwen_tokenizer", "deepseek_model", "deepseek_tokenizer"]:
    try:
        del globals()[name]
    except:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y transformers peft accelerate tokenizers huggingface_hub bitsandbytes trl
!pip install -q --no-cache-dir \
  "transformers==4.56.0" \
  "peft==0.17.1" \
  "accelerate==1.10.1" \
  "tokenizers==0.22.1" \
  "huggingface_hub==0.36.0" \
  "bitsandbytes==0.46.1" \
  "pyyaml" \
  "scikit-learn"

Found existing installation: transformers 4.48.2
Uninstalling transformers-4.48.2:
  Successfully uninstalled transformers-4.48.2
Found existing installation: peft 0.14.0
Uninstalling peft-0.14.0:
  Successfully uninstalled peft-0.14.0
Found existing installation: accelerate 1.2.1
Uninstalling accelerate-1.2.1:
  Successfully uninstalled accelerate-1.2.1
Found existing installation: tokenizers 0.21.0
Uninstalling tokenizers-0.21.0:
  Successfully uninstalled tokenizers-0.21.0
Found existing installation: huggingface-hub 0.27.1
Uninstalling huggingface-hub-0.27.1:
  Successfully uninstalled huggingface-hub-0.27.1
Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: trl 1.1.0
Uninstalling trl-1.1.0:
  Successfully uninstalled trl-1.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 157.1 MB/s e

In [ ]:
import transformers, peft, accelerate, tokenizers, huggingface_hub, bitsandbytes
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
print("tokenizers", tokenizers.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("bitsandbytes", bitsandbytes.__version__)

transformers 5.0.0
peft 0.18.1
accelerate 1.13.0
tokenizers 0.22.2
huggingface_hub 1.8.0
bitsandbytes 0.49.2


In [ ]:
!pip uninstall -y transformers peft accelerate tokenizers huggingface_hub bitsandbytes datasets pyarrow scikit-learn trl diffusers gradio sentence-transformers
!pip install -q --no-cache-dir \
  "transformers==5.0.0" \
  "peft==0.17.1" \
  "accelerate==1.10.1" \
  "tokenizers==0.22.1" \
  "huggingface_hub==0.36.0" \
  "bitsandbytes==0.46.1" \
  "pyyaml"

Found existing installation: transformers 4.56.0
Uninstalling transformers-4.56.0:
  Successfully uninstalled transformers-4.56.0
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
Found existing installation: accelerate 1.10.1
Uninstalling accelerate-1.10.1:
  Successfully uninstalled accelerate-1.10.1
Found existing installation: tokenizers 0.22.1
Uninstalling tokenizers-0.22.1:
  Successfully uninstalled tokenizers-0.22.1
Found existing installation: huggingface-hub 0.36.0
Uninstalling huggingface-hub-0.36.0:
  Successfully uninstalled huggingface-hub-0.36.0
Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
ERROR: Cannot install huggingface_hub==0.36.0 and transformers==5.0.0 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-wit

In [ ]:
from transformers import AutoConfig

deepseek_config = AutoConfig.from_pretrained(
    model_id_to_use,
    trust_remote_code=True,
)

print("Before:", deepseek_config.rope_parameters)

# patch default -> linear
deepseek_config.rope_parameters = {
    "rope_type": "linear",
    "factor": 1.0,
    "rope_theta": 10000.0,
}

deepseek_config.rope_scaling = {
    "rope_type": "linear",
    "factor": 1.0,
    "rope_theta": 10000.0,
}

print("After:", deepseek_config.rope_parameters)

Before: {'rope_type': 'default', 'rope_theta': 10000.0}
After: {'rope_type': 'linear', 'factor': 1.0, 'rope_theta': 10000.0}


In [ ]:
!pip install -q --no-cache-dir \
  "transformers==4.56.0" \
  "peft==0.17.1" \
  "accelerate==1.10.1" \
  "tokenizers==0.22.1" \
  "huggingface_hub==0.36.0" \
  "pyyaml"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 158.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 340.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 351.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchtune 0.6.1 requires datasets, which is not installed.
torchtune 0.6.1 requires sentencepiece, which is not installed.


In [ ]:
import transformers
print(transformers.__version__)

import transformers.models.llama.modeling_llama as m
print("OK:", m.__file__)

5.0.0
OK: /usr/local/lib/python3.12/dist-packages/transformers/models/llama/modeling_llama.py


In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_id_to_use,
    config=deepseek_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

print("Base model loaded successfully")

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Base model loaded successfully


In [ ]:
from peft import PeftModel

inference_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

inference_model.eval()

print("LoRA adapter loaded successfully")

LoRA adapter loaded successfully


In [ ]:
sample = val_rows[0]

messages = build_inference_messages(sample)

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=cfg.MAX_SEQ_LEN,
)
inputs = {k: v.to(next(inference_model.parameters()).device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )

pred_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Prediction:\n", pred_text)
print("\nTrue label:", sample["scenario_id"])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Prediction:
 The container failed to resolve the DNS name of a fictitious service in the cluster, leading to a DNS lookup failure. The Kubernetes Job failed because the container didn't produce the expected output.

True label: dns_resolution_failure


In [ ]:
from transformers.models.llama.modeling_llama import ROPE_INIT_FUNCTIONS

print("ROPE_INIT_FUNCTIONS keys:", ROPE_INIT_FUNCTIONS.keys())
print("'default' in keys:", "default" in ROPE_INIT_FUNCTIONS)

ROPE_INIT_FUNCTIONS keys: dict_keys(['linear', 'dynamic', 'yarn', 'longrope', 'llama3'])
'default' in keys: False


In [ ]:
correct = 0
unknown = 0
total = min(cfg.EVAL_SAMPLES, len(val_rows))

device = next(inference_model.parameters()).device

for i in range(total):
    sample = val_rows[i]
    messages = build_inference_messages(sample)

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=cfg.MAX_SEQ_LEN,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    pred_text = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    pred = predict_scenario_from_output(pred_text)
    true = sample["scenario_id"]

    if pred == "unknown":
        unknown += 1
    if pred == true:
        correct += 1

    if (i + 1) % 50 == 0:
        print(f"Processed {i+1}/{total}")

deepseek_metrics = {
    "accuracy": correct / total,
    "correct": correct,
    "total": total,
    "unknown": unknown,
}

print("\nDeepSeek Results")
print(f"Accuracy: {deepseek_metrics['accuracy'] * 100:.2f}%")
print(f"Correct:  {deepseek_metrics['correct']}/{deepseek_metrics['total']}")
print(f"Unknown:  {deepseek_metrics['unknown']}")

Processed 50/700
Processed 100/700
Processed 150/700
Processed 200/700
Processed 250/700
Processed 300/700
Processed 350/700
Processed 400/700
Processed 450/700
Processed 500/700
Processed 550/700
Processed 600/700
Processed 650/700
Processed 700/700

DeepSeek Results
Accuracy: 49.71%
Correct:  348/700
Unknown:  92


In [ ]:
print("\n===== FINAL COMPARISON =====")

print(f"Qwen Accuracy:      {qwen_metrics['accuracy'] * 100:.2f}%")
print(f"Qwen Correct:       {qwen_metrics['correct']}/{qwen_metrics['total']}")
print(f"Qwen Unknown:       {qwen_metrics['unknown']}")
print()

print(f"DeepSeek Accuracy:  {deepseek_metrics['accuracy'] * 100:.2f}%")
print(f"DeepSeek Correct:   {deepseek_metrics['correct']}/{deepseek_metrics['total']}")
print(f"DeepSeek Unknown:   {deepseek_metrics['unknown']}")


===== FINAL COMPARISON =====
Qwen Accuracy:      70.86%
Qwen Correct:       496/700
Qwen Unknown:       101

DeepSeek Accuracy:  49.71%
DeepSeek Correct:   348/700
DeepSeek Unknown:   92
